# RQ2 v2: capture once, replay, and record fresh-repeat variability

Compare **uncompressed LLaVA576**, **native input ApET288** and **the same native selection+basis without merging** on the fixed **GQA12 + MMStar12 + TextVQA6** RQ1 panel. Decoder compression stays off. This reviewed diagnostic cohort is not a population-accuracy sample; existing MMStar quality/grounding flags remain.

The original v1 pilot and its failed evidence remain unchanged. Its exact check compared independent native executions, but native merging itself varies numerically between executions. This v2 runner captures the **first native and first no-merge realization per image**, replays those fixed features for answers and scoring, and records fresh native repeats separately as uncertainty checks. Repeats never replace the first realization, select a more favorable answer, or approve the run by matching a tolerance. Native ApET source files, selection, basis placement and dependencies remain unchanged.

Push the new files to `apet-analysis`, import this notebook into an **editable draft**, and enable **Internet + T4 ×2**. Attach the **same private `rq1-ih-inputs-v1` Dataset** used previously: the existing 34.1 MiB ZIP or its extracted files. No new upload is needed; the older `rq1-inputs-v1` alone is insufficient.

**First run only the seven-question calibration** with `RUN_V2_CALIBRATION = True`. Leave `RUN_V2_FULL = False`. Download its ZIP/log for review before running the full panel. Full mode needs an explicit successful calibration `report.json` and its detailed evidence files. Both flags default to False and stop before cloning, setup or inference. See `research/RQ2_INPUT_MERGE_V2.md` for instructions, including a cell for an already running notebook.


In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

RUN_V2_CALIBRATION = False  # First: seven predeclared questions across all three datasets.
RUN_V2_FULL = False         # Later: all 30, after reviewing the calibration ZIP/log.
CALIBRATION_REPORT = None   # Full mode only: exact path to calibration report.json, with details present.
REPO_URL = "https://github.com/doguineyes/CSE590-ApET.git"
REPO_REF = "apet-analysis"
INPUT_ZIP = None            # Optional exact ZIP path OR extracted folder containing references/.
CHECKPOINT_ARGS = []        # Optional ["--model", "/kaggle/input/...", "--vision-model", "/kaggle/input/..."]

if RUN_V2_CALIBRATION and RUN_V2_FULL:
    raise ValueError("Enable only one mode: v2 calibration OR full 30-question panel.")
if not (RUN_V2_CALIBRATION or RUN_V2_FULL):
    raise RuntimeError("Enable RUN_V2_CALIBRATION first, Internet + T4 x2, then execute cells manually.")
if RUN_V2_FULL and CALIBRATION_REPORT is None:
    raise ValueError("Full mode requires CALIBRATION_REPORT and its detailed calibration evidence.")
if RUN_V2_FULL and not Path(CALIBRATION_REPORT).is_file():
    raise FileNotFoundError("CALIBRATION_REPORT must point to an existing calibration report.json.")
print("Preparing RQ2 v2 code; model inference has not started.", flush=True)
repo = Path("/kaggle/working/CSE590-ApET")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True, timeout=120)
subprocess.run(["git", "fetch", "origin"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "checkout", REPO_REF], cwd=repo, check=True, timeout=120)
if REPO_REF == "apet-analysis":
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True, timeout=120)
if not (repo / "scripts/run_rq2_input_merge_v2.py").is_file():
    raise RuntimeError("Push the new RQ2 v2 files first, then update this checkout.")
original_python = repo / "environments/original/.venv/bin/python"
rq2_v2_runs = []
print("Repository:", repo, flush=True)


## Prepare existing inputs and locked original runtime

This reuses the original isolated environment and checkpoint cache within a live session. Kaggle's notebook kernel remains unchanged. A new session may need setup and checkpoint downloads again. The next helper cell runs tiny native tests before pretrained loading and archives the setup/native-test log and runtime report. A zero-test run or any skipped test stops execution.


In [ ]:
sys.path.insert(0, str(repo / "scripts"))
from rq1_input_ih import locate_inputs
inputs = locate_inputs(
    [Path("/kaggle/input"), Path("/kaggle/working")],
    Path("/kaggle/working/rq1-ih-inputs-v1"), explicit=INPUT_ZIP,
)
print("Accepted input directory:", inputs, flush=True)
setup_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
setup_log = Path(f"/kaggle/working/rq2-v2-setup-{setup_stamp}.log")
runtime_report = Path(f"/kaggle/working/rq2-v2-runtime-{setup_stamp}.json")

def stream_command(command, stream, *, cwd=None):
    print("Running:", " ".join(map(str, command)), flush=True)
    stream.write("COMMAND " + repr(command) + "\n")
    stream.flush()
    process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8", bufsize=1)
    lines = []
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            stream.write(line)
            stream.flush()
            lines.append(line)
        status = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    stream.write(f"EXIT {status}\n")
    stream.flush()
    if status:
        raise RuntimeError(f"Command failed (exit {status}); preserve {stream.name} before changing anything.")
    return "".join(lines)

with setup_log.open("x", encoding="utf-8") as stream:
    stream_command([
        "bash", "bootstrap/setup_original_linux.sh", "--require-cuda", "--expected-gpus", "2",
        "--output", str(runtime_report),
    ], stream, cwd=repo)
print("Locked original runtime ready. Run the prerequisite/helper cell next.", flush=True)
print("Setup log:", setup_log)
print("Runtime report:", runtime_report)


## Define the helper and verify native prerequisites

The helper creates a new directory for each invocation. Original native selection, E, basis source-to-slot placement and merge assignments are checked. Answers and reference scoring replay the captured first realization; fresh repeats are recorded separately and do not replace it. A calibration is not the full panel, and numerical variation is evidence rather than a reason to choose a different seed or answer.

The runner saves evidence on normally handled failures. This wrapper leaves the runner's ZIP untouched, saves the console log separately, and creates a labeled fallback only if no ZIP was produced. Hard session termination can prevent cleanup. Do not pull Git or start another model process while inference runs.


In [ ]:
def verify_rq2_v2_prerequisites(*, refresh_runtime=False):
    """Run tiny native tests; refresh provenance without reinstalling in a live session."""
    with setup_log.open("a", encoding="utf-8") as stream:
        if refresh_runtime:
            stream_command([
                str(original_python), "-X", "utf8", "-I", "scripts/check_original_env.py",
                "--require-cuda", "--expected-gpus", "2", "--output", str(runtime_report),
            ], stream, cwd=repo)
        # Import Torch explicitly so Kaggle cannot take the local no-Torch skip path.
        stream_command([str(original_python), "-X", "utf8", "-I", "-c",
                        "import torch; print(torch.__version__)"], stream, cwd=repo)
        native_output = stream_command([
            str(original_python), "-X", "utf8", "-m", "unittest",
            "research.tests.test_rq2_native",
            "research.tests.test_rq2_frozen_native", "-v",
        ], stream, cwd=repo)
        count = re.search(r"Ran (\d+) tests?", native_output)
        if not count or int(count.group(1)) == 0 or re.search(r"skipped\s*=", native_output):
            raise RuntimeError("Native prerequisites must run with zero skips; preserve the setup log.")
    print("RQ2 v2 native prerequisites passed.", flush=True)


def run_rq2_v2(*, calibration=True, calibration_report=None):
    if not isinstance(calibration, bool):
        raise TypeError("calibration must be True or False.")
    if calibration and calibration_report is not None:
        raise ValueError("Calibration mode does not take a previous calibration report.")
    if not calibration:
        if calibration_report is None:
            raise ValueError("Full mode requires the reviewed calibration report.json and detailed evidence.")
        calibration_report = Path(calibration_report)
        if not calibration_report.is_file():
            raise FileNotFoundError(calibration_report)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    prefix = "rq2-input-merge-v2-calibration" if calibration else "rq2-input-merge-v2-full"
    folder = Path(f"/kaggle/working/{prefix}-{stamp}")
    log = folder.with_suffix(".log")
    archive = folder.with_suffix(".zip")
    globals().setdefault("rq2_v2_runs", []).append(folder)
    command = [str(original_python), "-X", "utf8", "-I", "scripts/run_rq2_input_merge_v2.py",
        "--inputs", str(inputs), "--output", str(folder),
        "--require-cuda", "--expected-gpus", "2", "--download-budget-gib", "20",
        "--setup-log", str(setup_log), "--runtime-report", str(runtime_report), *CHECKPOINT_ARGS]
    if calibration:
        command.append("--calibration")
    else:
        command.extend(["--calibration-report", str(calibration_report)])
    print("Mode:", "seven-question v2 calibration" if calibration else "full 30-question v2 panel", flush=True)
    print("Run directory:", folder, flush=True)
    fallback_archive = False
    try:
        with log.open("x", encoding="utf-8") as stream:
            stream_command(command, stream, cwd=repo)
    finally:
        # Normal handled failures already have a complete runner evidence ZIP.
        if not archive.exists():
            fallback_archive = True
            folder.mkdir(parents=True, exist_ok=True)
            fallback = folder / "notebook-fallback"
            fallback.mkdir(exist_ok=True)
            (fallback / "README.txt").write_text(
                "Notebook fallback evidence: the process did not produce its normal ZIP.\n"
                "This archive does not establish successful execution or complete results.\n",
                encoding="utf-8")
            for source in [setup_log, runtime_report, log]:
                if source.is_file():
                    shutil.copy2(source, fallback / source.name)
            with zipfile.ZipFile(archive, "x", zipfile.ZIP_DEFLATED) as bundle:
                for source in sorted(folder.rglob("*")):
                    if source.is_file():
                        bundle.write(source, source.relative_to(folder))
            print("Created incomplete fallback evidence ZIP:", archive, flush=True)
        print("Download:", archive, "and", log, flush=True)
        report_path = folder / "report.json"
        if report_path.exists():
            try:
                report = json.loads(report_path.read_text(encoding="utf-8"))
                print("Status:", report.get("status"), "mode:", report.get("mode"),
                      "wall seconds:", report.get("wall_seconds"), flush=True)
                for name in ["progress", "summaries", "error"]:
                    if report.get(name) is not None:
                        print(name + ":", report[name], flush=True)
            except (ValueError, OSError) as error:
                print("Could not read final report:", error, flush=True)
    if fallback_archive:
        raise RuntimeError("Runner archive missing; preserve the fallback ZIP/log before rerunning.")
    if not (folder / "report.json").is_file():
        raise RuntimeError("No runner report was produced; preserve the ZIP/log.")
    report = json.loads((folder / "report.json").read_text(encoding="utf-8"))
    if report.get("version") != "rq2-input-merge-v2":
        raise RuntimeError("Unexpected runner version; preserve the ZIP/log.")
    if report.get("mode") != ("calibration" if calibration else "full"):
        raise RuntimeError("Runner mode mismatch; preserve the ZIP/log.")
    if report.get("status") != "passed":
        raise RuntimeError("RQ2 v2 run did not pass; download the ZIP/log for review.")
    if calibration:
        print("Calibration completed. Download ZIP/log for review before starting the full panel.", flush=True)
        print("Calibration report:", folder / "report.json", flush=True)
    return folder


In [ ]:
verify_rq2_v2_prerequisites()


## Run only the selected mode

Start with the seven predeclared calibration questions: a GQA same-image pair, one MMStar example and four TextVQA examples, including literal-text loss cases and a preserved control. They exercise the capture/replay and fresh-repeat checks. Do not change the subset in response to its outcomes.

After calibration review, full mode runs all 30 frozen questions. Set `RUN_V2_CALIBRATION = False`, `RUN_V2_FULL = True`, and `CALIBRATION_REPORT` to the calibration `report.json`. Keep that report's entire output directory, including detailed records, available. The runner validates this evidence before the full run. Existing calibration files in the same live session can be reused.


In [ ]:
if RUN_V2_CALIBRATION:
    rq2_v2_dir = run_rq2_v2(calibration=True)
elif RUN_V2_FULL:
    rq2_v2_dir = run_rq2_v2(calibration=False, calibration_report=CALIBRATION_REPORT)
else:
    raise RuntimeError("Enable one mode in the first cell before running.")


## Download before stopping Kaggle

Download the **result ZIP and matching console log**, including after failures. The setup log and runtime JSON are included in normal runner evidence and also linked below. Send the calibration ZIP/log for review before proceeding. Keep v1 failed archives and the nine-pass diagnostic unchanged.

Each run reloads the model while reusing the environment and checkpoints within this session. Separate notebooks do not share a virtualenv/session; they can attach the same private input Dataset. Preserve the complete calibration ZIP if the session may end: full mode needs its extracted report and detailed records, not the standalone report alone.


In [ ]:
from IPython.display import FileLink, display
paths = [setup_log, runtime_report]
for folder in rq2_v2_runs:
    paths.extend([folder.with_suffix(".zip"), folder.with_suffix(".log")])
for path in paths:
    if path.is_file():
        print(path)
        display(FileLink(str(path)))
